# What does cross-validation actually decide?

`RidgeClassifierCV` fits a classifier and picks its regularization by
leave-one-out cross-validation. This notebook traces one fit and reads
the decision as mathematics: which branches the fit took, under what
conditions the chosen alpha wins, and what happens to samples sitting
exactly on the class boundary.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import sys
sys.path.insert(0, "../skverify-hypothesis")

import numpy as np
import sympy
from sklearn.linear_model import RidgeClassifierCV
from skverify_hypothesis import explore, edge_cases, verify

def fit_coef(X, yraw):
    y = (yraw > 0.0).astype(float)
    return RidgeClassifierCV(alphas=[0.1, 1.0, 10.0]).fit(X, y).coef_.ravel()

rng = np.random.default_rng(0)
X = rng.standard_normal((8, 2)); yraw = rng.standard_normal(8)
out = verify(fit_coef, X, yraw)
out.formula

_solve_eigen_covariance_6_1[i, 0]

The coefficients come back as a named term: the eigendecomposition
solve the fit ran, checked against its own equation on this call. The
interesting part is the assumptions. Two of them are the
cross-validation decision itself:

In [2]:
from IPython.display import Math, display
for g in out.preconditions.args:
    if "solve_eigen" in str(g):
        display(Math(sympy.latex(g)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Read them in plain words. Each named term holds the leave-one-out
residuals for one candidate alpha. The two inequalities say: the mean
squared error of alpha 2 beat alpha 1, and alpha 3 beat alpha 1. These
coefficients are valid exactly as long as those comparisons hold. Flip
one by changing the data and the model would have picked a different
alpha.

The other assumptions record the class each sample got:

In [3]:
for g in list(out.preconditions.args)[:3]:
    if "yraw" in str(g):
        display(Math(sympy.latex(g)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Now let the machine find the branches instead of us. Thirty random
draws, one witness kept per distinct set of assumptions:

In [4]:
paths = explore(fit_coef, (np.zeros((8, 2)), np.zeros(8)), max_examples=30)
len(paths)

30

Around twenty distinct computational paths for one estimator at
n = 8: every label split and every alpha winner is its own branch,
each with its own formula and its own conditions.

Last: the inputs most likely to cause trouble. Every assumption has a
boundary, and `edge_cases` builds inputs sitting exactly on them. Here
that means samples exactly on the class threshold:

In [5]:
cases = edge_cases(fit_coef, (np.zeros((8, 2)), np.zeros(8)), paths=paths[:3])
for g, args, outcome in cases[:4]:
    which = str(g).split("yraw")[1][:3] if "yraw" in str(g) else "?"
    print(f"sample yraw{which} put exactly on the threshold -> coef {np.round(outcome, 4)}")

sample yraw[0] put exactly on the threshold -> coef [-0.0917 -0.2093]
sample yraw[1] put exactly on the threshold -> coef [ 0.1639 -0.1658]
sample yraw[2] put exactly on the threshold -> coef [-0.0859 -0.2361]
sample yraw[3] put exactly on the threshold -> coef [-0.0859 -0.2361]


**Takeaway.** One trace turned a cross-validated fit into a formula
plus its conditions. The alpha choice is two inequalities you can
check. The branches are countable and the boundary inputs are
constructable. None of this needed reading sklearn's source.